In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [3]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [4]:
apps_df = pd.read_csv('Play store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')

In [5]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [6]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [7]:
#data Cleaning
apps_df = apps_df.dropna(subset=['Rating'])
for column in apps_df.columns:
    try:
        apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)
    except Exception:
        # fallback to forward fill if mode() fails (e.g., all NaN)
        apps_df[column].fillna(method='ffill', inplace=True)
apps_df.drop_duplicates(inplace=True)
# keep ratings <= 5 and ensure correct assignment
apps_df = apps_df[apps_df['Rating'] <= 5]
# drop rows in reviews_df where Translated_Review is missing
reviews_df = reviews_df.dropna(subset=['Translated_Review'])

C:\Users\HP\AppData\Local\Temp\ipykernel_20308\2525615808.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)


In [8]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [9]:
#Transforming Data Types
# Converting install column to numeric by removing '+' and ',' characters
apps_df['Installs'] = apps_df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(int)

# Converting Price column to numeric by removing '$' character
apps_df['Price'] = apps_df['Price'].str.replace('$', '', regex=False).astype(float)

In [10]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs            int64
Type               object
Price             float64
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [11]:
merged_df = pd.merge(reviews_df, apps_df, on='App', how='inner')# inner join to keep only matching records

In [12]:
merged_df

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.000000,0.533333,HEALTH_AND_FITNESS,4.0,2490,3.8M,500000,Free,0.0,Everyone 10+,Health & Fitness,"February 17, 2017",1.9,2.3.3 and up
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.250000,0.288462,HEALTH_AND_FITNESS,4.0,2490,3.8M,500000,Free,0.0,Everyone 10+,Health & Fitness,"February 17, 2017",1.9,2.3.3 and up
2,10 Best Foods for You,Works great especially going grocery store,Positive,0.400000,0.875000,HEALTH_AND_FITNESS,4.0,2490,3.8M,500000,Free,0.0,Everyone 10+,Health & Fitness,"February 17, 2017",1.9,2.3.3 and up
3,10 Best Foods for You,Best idea us,Positive,1.000000,0.300000,HEALTH_AND_FITNESS,4.0,2490,3.8M,500000,Free,0.0,Everyone 10+,Health & Fitness,"February 17, 2017",1.9,2.3.3 and up
4,10 Best Foods for You,Best way,Positive,1.000000,0.300000,HEALTH_AND_FITNESS,4.0,2490,3.8M,500000,Free,0.0,Everyone 10+,Health & Fitness,"February 17, 2017",1.9,2.3.3 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59119,Housing-Real Estate & Property,Most ads older many agents ..not much owner po...,Positive,0.173333,0.486667,LIFESTYLE,4.1,28301,Varies with device,1000000,Free,0.0,Everyone,Lifestyle,"July 13, 2018",12.1.0,4.1 and up
59120,Housing-Real Estate & Property,"If photos posted portal load, fit purpose. I'm...",Positive,0.225000,0.447222,LIFESTYLE,4.1,28301,Varies with device,1000000,Free,0.0,Everyone,Lifestyle,"July 13, 2018",12.1.0,4.1 and up
59121,Housing-Real Estate & Property,"Dumb app, I wanted post property rent give opt...",Negative,-0.287500,0.250000,LIFESTYLE,4.1,28301,Varies with device,1000000,Free,0.0,Everyone,Lifestyle,"July 13, 2018",12.1.0,4.1 and up
59122,Housing-Real Estate & Property,I property business got link SMS happy perform...,Positive,0.800000,1.000000,LIFESTYLE,4.1,28301,Varies with device,1000000,Free,0.0,Everyone,Lifestyle,"July 13, 2018",12.1.0,4.1 and up


In [13]:
#data Transformation
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'K' in size:
        return float(size.replace('K',''))/1024
    else:
        return np.nan
apps_df['Size']= apps_df['Size'].apply(convert_size)

In [14]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [15]:
#logrithmic
apps_df['Log_Installs'] = np.log1p(apps_df['Installs'])

In [16]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size              float64
Installs            int64
Type               object
Price             float64
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
Log_Installs      float64
dtype: object

In [17]:
apps_df['Reviews'] = apps_df['Reviews'].astype(int)

In [18]:
apps_df['Log_Reviews'] = np.log1p(apps_df['Reviews'])

In [19]:
#rating groups
def rating_group(rating):
    if rating >= 4.0:
        return 'top rated app'
    elif rating >= 3.0:
        return 'above average'
    elif rating >= 2.0:
        return 'average'
    else:
        return 'below average'
apps_df['Rating_Group'] = apps_df['Rating'].apply(rating_group)

In [20]:
apps_df['Rating_Group'].head()


0    top rated app
1    above average
2    top rated app
3    top rated app
4    top rated app
Name: Rating_Group, dtype: object

In [21]:
#deriving matrics revenue column
apps_df['Revenue'] = apps_df['Price'] * apps_df['Installs']

In [22]:
#sentiment analysis
sia = SentimentIntensityAnalyzer()

In [23]:
#polarity score in Sia
#positive negative neutral compound -1: very negative 1: very positive
reviews_df['Sentiment_Score'] = reviews_df['Translated_Review'].apply(lambda x: sia.polarity_scores(x)['compound'])


In [24]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [25]:
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'] ,errors='coerce') 

In [26]:
apps_df['Year'] = apps_df['Last Updated'].dt.year

In [27]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210440,5.075174,top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122365,6.875232,above average,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424949,11.379520,top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281389,top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512935,6.875232,top rated app,0.0,2018


In [28]:
html_file_path = "./"
if not os.path.exists(html_file_path):
    os.makedirs(html_file_path)

In [29]:
plot_containers ="" 

In [30]:
def save_plot_html(fig, filename, insight):
    """Save a plotly figure as HTML with given filename and add it to the container"""
    global plot_containers
    filepath = os.path.join(html_file_path, filename)
    html_content = pio.to_html(fig, include_plotlyjs='cdn', full_html=False)
    
    # Write the plot to a file
    fig.write_html(filepath, include_plotlyjs='cdn')
    
    # Add to the plot containers
    plot_containers += f"""<div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
    <div class="plot">{html_content}</div>
    <div class="insight"><h3>Insight:</h3><p>{insight}</p></div>
    </div>"""
    fig.write_html(filepath,full_html = "False", include_plotlyjs='inline')


In [31]:
plot_width=400
plot_height=300
plot_bg_color='black'
text_color='white'
title_font={'size':16}
axis_font={'size':12}

In [32]:
#figure1
category_counts = apps_df['Category'].value_counts().nlargest(10)
fig1 = px.bar(
    x=category_counts.index,
    y=category_counts.values,
    title="Top 10 App Categories on play store",
    labels={"x": "Category", "y": "Number of Apps"},
    color=category_counts.index,
    color_discrete_sequence= px.colors.sequential.Plasma,
    width=400,
    height=300
)
fig1.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig1.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig1,"category graph 1.html", "The top categories on the Play Store are dominated by tools, entertainment, and productivity apps")

In [33]:
#figure2 pie chart of paid and free apps
type_counts = apps_df['Type'].value_counts()
fig2 = px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title="paid vs free apps",
    color_discrete_sequence= px.colors.sequential.RdBu,
    width=400,
    height=300
)
fig2.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    margin=dict(l=40, r=40, t=40, b=40)
)
#fig1.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig2,"type.html", "number of paid vs free apps")

In [34]:
#figure3 rating distribution histogram
fig3 = px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution of Apps',
    color_discrete_sequence=['#636efa'],
    width=400,
    height=300
)
fig3.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
fig3.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig3,"rating_distribution.html", "distribution of app ratings on play store")

In [35]:
#figure4 sentiment score distribution bar graph
setiment_counts = reviews_df['Sentiment_Score'].value_counts()
fig4 = px.bar(
    x=setiment_counts.index,
    y=setiment_counts.values,
    title="setiment Score play store Apps",
    labels={"x": "Sentiment Score", "y": "Counts"},
    color=setiment_counts.index,
    color_discrete_sequence= px.colors.sequential.RdPu,
    width=400,
    height=300
)
fig4.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
fig4.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig4,"Sentiment_Score.html", "Sentiment on play store apps")

In [36]:
#figure5 genre
installs_genre = apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5 = px.bar(
    x=installs_genre.index,
    y=installs_genre.values,
    title="Top 10 Genres by Total Installs",
    labels={"x": "Genre", "y": "Total Installs"},
    color=installs_genre.index,
    color_discrete_sequence= px.colors.sequential.Viridis,
    width=400,
    height=300
)
fig5.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
fig5.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig5,"top_genres.html", "top 10 app genres by total installs on play store")


In [37]:
#figure 6
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    title="Number of App Updates Per Year",
    labels={"x": "Year", "y": "Number of Updates"},
    color_discrete_sequence= px.colors.sequential.Teal,
    width=400,
    height=300
)
fig6.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
fig6.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig6,"app_updates.html", "number of app updates per year on play store")

In [38]:
#figure 7 Revenue by Category
revenue_by_category = apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7 = px.bar(
    x=revenue_by_category.index,
    y=revenue_by_category.values,
    title="Top 10 Categories by Revenue",
    labels={"x": "Category", "y": "Total Revenue"},
    color=revenue_by_category.index,
    color_discrete_sequence= px.colors.sequential.Magenta,
    width=400,
    height=300
)
fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
fig7.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig7,"revenue_by_category.html", "top 10 app categories by revenue on play store")

In [39]:
# figure 8 top 10 genres 
genre_counts = apps_df['Genres'].str.split(';' ,expand=True).stack().value_counts().nlargest(10)
fig8 = px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    title="Top 10 Genres on Play Store",
    labels={"x": "Genre", "y": "Number of Apps"},
    color=genre_counts.index,
    color_discrete_sequence= px.colors.sequential.Cividis,
    width=400,
    height=300
)
fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
fig8.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig8,"top_genres_overall.html", "top 10 app genres on play store overall")

In [40]:
#figure 9 Last Upadate on app rating based on free and paid apps
fig9 = px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='App Ratings Over Time by Type',
    color_discrete_sequence= px.colors.qualitative.Vivid,
    width=400,
    height=300
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
#fig9.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig9,"ratings_over_time.html", "app ratings over time on play store")

In [41]:
#figure 10 Rating of paid vs free apps by box plot
fig10 = px.box(
    apps_df,
    x='Type',
    y='Rating',
    title='Rating Distribution by App Type',
    color_discrete_sequence= px.colors.qualitative.Pastel,
    width=400,
    height=300
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    margin=dict(l=40, r=40, t=40, b=40)
)
fig10.update_traces(marker_line_color='white', marker_line_width=1.5)
save_plot_html(fig10,"rating_boxplot.html", "rating distribution of paid vs free apps on play store")

In [66]:
#Use a grouped bar chart to compare the average rating and total review count for the top 10 app categories by number of installs.
#  Filter out any categories where the average rating is below 4.0 and size below 10 M and last update should be Jan month . 
# this graph should work only between 3PM IST to 5 PM IST apart from that time we should not show this graph in dashboard itself.
from datetime import datetime
import pytz
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)
if current_time.hour >= 15 and current_time.hour < 17:
    filtered_apps = apps_df[
        (apps_df['Rating'] >= 4.0) &
        (apps_df['Size'] >= 10) &
        (apps_df['Last Updated'].dt.month == 1)
    ]
    if filtered_apps.empty:
        print('No apps meet the filtering criteria.')
    else:
        top_categories = (
            filtered_apps
            .groupby('Category')
            .agg(Installs=('Installs','sum'), Rating=('Rating','mean'), Reviews=('Reviews','sum'))
            .reset_index()
        )
        top_categories = top_categories.nlargest(10, 'Installs')
        #top_categories = top_categories.sort_values(by='Installs', ascending=False).head(10)
        if top_categories.empty:
            print('No categories after aggregation.')
        else:
            fig = px.bar(
                top_categories,
                x='Category',
                y=['Rating', 'Reviews'],
                barmode='group',
                title='Average Rating and Total Reviews for Top 10 App Categories by Installs',
                labels={'value': 'Count', 'Category': 'App Category'},
                color_discrete_sequence= px.colors.qualitative.Dark24,
                width=400,
                height=300
            )
            fig.update_layout(
                plot_bgcolor='black',
                paper_bgcolor='black',
                font_color='white',
                title_font_size=20,
                xaxis_title_font_size=16,
                yaxis_title_font_size=16,
                margin=dict(l=40, r=40, t=40, b=40)
            )
            fig.update_traces(marker_line_color='white', marker_line_width=1.5)
            save_plot_html(fig1,"top_categories_time_based.html", "average rating and total reviews for top 10 app categories by installs on play store")


In [65]:
#Create an interactive Choropleth map using Plotly to visualize global installs by Category. Apply filters to show data for only
#  the top 5 app categories and highlight category where the number of installs exceeds 1 million. 
# The app category should not start with the characters “A,” “C,” “G,” or “S.” This graph should work only between 6 PM IST 
# and 8 PM IST; apart from that time, we should not show it in the dashboard itself.
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)
if current_time.hour >= 18 and current_time.hour < 20:
    excluded_initials = ('A', 'C', 'G', 'S')
    filtered_apps = apps_df[~apps_df['Category'].str.startswith(excluded_initials)]
    
    category_installs = (
        filtered_apps
        .groupby('Category')
        .agg(Total_Installs=('Installs', 'sum'))
        .reset_index()
    )
    
    top_5_categories = category_installs.nlargest(5, 'Total_Installs')
    
    if top_5_categories.empty:
        print('No categories meet the filtering criteria.')
    else:
        fig2 = px.choropleth(
            top_5_categories,
            locations='Category',
            locationmode='country names',
            color='Total_Installs',
            hover_name='Category',
            color_continuous_scale=px.colors.qualitative.Vivid,
            title='Global Installs by Top 5 App Categories',
            width=400,
            height=300
        )
        
        fig2.update_layout(
            plot_bgcolor='black',
            paper_bgcolor='black',
            font_color='white',
            title_font_size=20,
            xaxis_title_font_size=16,
            yaxis_title_font_size=16,
            margin=dict(l=40, r=40, t=40, b=40)
        )
        
        fig2.update_traces(marker_line_color='white', marker_line_width=1.5)
        
        save_plot_html(fig2, "choropleth_top_categories_time_based.html", "global installs by top 5 app categories on play store")


In [64]:

#Create a dual-axis chart comparing the average installs and revenue for free vs. paid apps within the top 3 app categories.
#  Apply filters to exclude apps with fewer than 10,000 installs and revenue below $10,000 and android version should be 
# more than 4.0 as well as size should be more than 15M and content rating should be Everyone and app name should not have more 
# than 30 characters including space and special character .this graph should work only between 1 PM IST to 2 PM IST apart from 
# that time we should not show this graph in dashboard itself.
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)
if current_time.hour >= 13 and current_time.hour < 14:
    filtered_apps = apps_df[
        (apps_df['Installs'] >= 10000) &
        (apps_df['Revenue'] >= 10000) &
        (apps_df['Android Ver'].apply(lambda x: float(x.split('.')[0]) if isinstance(x, str) and x.split('.')[0].isdigit() else 0) > 4.0) &
        (apps_df['Size'] > 15) &
        (apps_df['Content Rating'] == 'Everyone') &
        (apps_df['App'].apply(lambda x: len(x) <= 30))
    ]
    
    top_3_categories = (
        filtered_apps
        .groupby('Category')
        .agg(Total_Installs=('Installs', 'sum'))
        .reset_index()
        .nlargest(3, 'Total_Installs')
    )['Category']
    
    comparison_data = filtered_apps[filtered_apps['Category'].isin(top_3_categories)]
    
    avg_metrics = (
        comparison_data
        .groupby(['Category', 'Type'])
        .agg(Average_Installs=('Installs', 'mean'), Average_Revenue=('Revenue', 'mean'))
        .reset_index()
    )
    
    if avg_metrics.empty:
        print('No data available after applying filters.')
    else:
        fig3 = px.bar(
            avg_metrics,
            x='Category',
            y='Average_Installs',
            color='Type',
            barmode='group',
            title='Average Installs for Free vs Paid Apps in Top 3 Categories',
            labels={'Average_Installs': 'Average Installs', 'Category': 'App Category'},
            width=400,
            height=300
        )
        
        fig3.update_layout(
            plot_bgcolor='black',
            paper_bgcolor='black',
            font_color='white',
            title_font_size=20,
            xaxis_title_font_size=16,
            yaxis_title_font_size=16,
            margin=dict(l=40, r=40, t=40, b=40)
        )
        
        fig3.update_traces(marker_line_color='white', marker_line_width=1.5)
        
        save_plot_html(fig3, "dual_axis_installs_revenue_time_based.html", "average installs and revenue for free vs paid apps in top 3 categories on play store  ")

In [45]:
#Plot a time series line chart to show the trend of total installs over time, segmented by app category. 
# Highlight periods of significant growth by shading the areas under the curve where the increase in installs exceeds 
# 20% month-over-month and app name should not starts with x, y ,z and app category should start with letter " E " or " C " or 
# " B " and We have to translate the Beauty category in Hindi and Business category in Tamil and Dating category in German while 
# showing it on Graph. reviews should be more than 500 the app name should not contain letter "S" as well as this graph should 
# work only between 6 PM IST to 9 PM IST apart from that time we should not show this graph in dashboard itself
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)
if current_time.hour >= 18 and current_time.hour < 21:
    filtered_apps = apps_df[
        (apps_df['Reviews'] > 500) &
        (~apps_df['App'].str.contains('S', case=False)) &
        (~apps_df['App'].str.startswith(('X', 'Y', 'Z'))) &
        (apps_df['Category'].str.startswith(('E', 'C', 'B')))
    ]
    
    # Translate categories
    category_translation = {
        'Beauty': 'सौंदर्य',
        'Business': 'வணிகம்',
        'Dating': 'Dating'  # Assuming no translation needed for Dating in German
    }
    
    filtered_apps['Translated_Category'] = filtered_apps['Category'].replace(category_translation)
    
    # Aggregate installs by month and category
    filtered_apps['Month'] = filtered_apps['Last Updated'].dt.to_period('M')
    monthly_installs = (
        filtered_apps
        .groupby(['Month', 'Translated_Category'])
        .agg(Total_Installs=('Installs', 'sum'))
        .reset_index()
    )
    
    monthly_installs['Month'] = monthly_installs['Month'].dt.to_timestamp()
    
    fig4 = px.line(
        monthly_installs,
        x='Month',
        y='Total_Installs',
        color='Translated_Category',
        title='Trend of Total Installs Over Time by App Category',
        labels={'Total_Installs': 'Total Installs', 'Month': 'Month'},
        width=400,
        height=300
    )
    
    fig4.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font_size=20,
        xaxis_title_font_size=16,
        yaxis_title_font_size=16,
        margin=dict(l=40, r=40, t=40, b=40)
    )
    
    fig4.update_traces(marker_line_color='white', marker_line_width=1.5)
    
    save_plot_html(fig4, "time_series_installs_by_category_time_based.html", "trend of total installs over time by app category on play store")

C:\Users\HP\AppData\Local\Temp\ipykernel_20308\743616197.py:24: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\HP\AppData\Local\Temp\ipykernel_20308\743616197.py:27: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [46]:
print("Total apps:", apps_df.shape[0])
print("Rating > 3.5:", apps_df[apps_df['Rating'] > 3.5].shape[0])
print("Reviews > 500:", apps_df[apps_df['Reviews'] > 500].shape[0])
print("Installs > 50000:", apps_df[apps_df['Installs'] > 50000].shape[0])
print("No 'S' in name:", apps_df[~apps_df['App'].str.contains('S', case=False)].shape[0])
print("Selected categories:", apps_df[apps_df['Category'].isin(['Game','Beauty','Business','Comics','Communication','Dating','Entertainment','Social','Event'])].shape[0])


Total apps: 8892
Rating > 3.5: 8012
Reviews > 500: 5949
Installs > 50000: 5678
No 'S' in name: 3207
Selected categories: 0


In [47]:
apps_df['Category'] = apps_df['Category'].astype(str).str.strip().str.lower()

selected_categories = [
    'game', 'beauty', 'business', 'comics', 
    'communication', 'dating', 'entertainment', 'social', 'event'
]

filtered_apps = apps_df[
    (apps_df['Rating'] > 3.5) &
    (apps_df['Reviews'] > 500) &
    (apps_df['Installs'] > 50000) &
    (~apps_df['App'].str.contains('S', case=False)) &
    (apps_df['Category'].isin(selected_categories))
]

print(apps_df['Category'].unique())



['art_and_design' 'auto_and_vehicles' 'beauty' 'books_and_reference'
 'business' 'comics' 'communication' 'dating' 'education' 'entertainment'
 'events' 'finance' 'food_and_drink' 'health_and_fitness' 'house_and_home'
 'libraries_and_demo' 'lifestyle' 'game' 'family' 'medical' 'social'
 'shopping' 'photography' 'sports' 'travel_and_local' 'tools'
 'personalization' 'productivity' 'parenting' 'weather' 'video_players'
 'news_and_magazines' 'maps_and_navigation']


In [63]:
#change
#Plot a bubble chart to analyze the relationship between app size (in MB) and average rating, with the bubble size representing the number of installs. 
# Include a filter to show only apps with a rating higher than 3.5 and that belong to the Game, Beauty ,business , commics , commication , Dating , Entertainment , 
# social and event categories. Reviews should be greater than 500 and the app name should not contain letter "S" and highlight the Game Category chart in Pink color. 
# We have to translate the Beauty category in Hindi and Business category in Tamil and Dating category in German while showing it on Graphs. 
# Installs should be more than 50k as well as this graph should work only between 5 PM IST to 7 PM IST.
# --- Import required libraries ---

import pandas as pd
import plotly.express as px
from datetime import datetime
import pytz

# Set IST timezone
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)

# Check time condition
if 17 <= current_time.hour < 19:
    # Convert category to lowercase and trim spaces
    apps_df['Category'] = apps_df['Category'].astype(str).str.strip().str.lower()
    
    # Filter conditions
    selected_categories = [
        'game', 'beauty', 'business', 'comics',
        'communication', 'dating', 'entertainment', 'social', 'events'
    ]
    
    filtered_apps = apps_df[
        (apps_df['Rating'] > 3.5) &
        (apps_df['Reviews'] > 500) &
        (apps_df['Installs'] > 50000) &
        (~apps_df['App'].str.contains('S', case=False)) &
        (apps_df['Category'].isin(selected_categories))
    ]
    
    if not filtered_apps.empty:
        # Translate category names
        translation = {
            'beauty': 'सौंदर्य',      # Hindi
            'business': 'வணிகம்',   # Tamil
            'dating': 'Verabredung'  # German
        }
        filtered_apps['Translated_Category'] = filtered_apps['Category'].map(lambda x: translation.get(x, x))
        
        # Create bubble chart
        fig = px.scatter(
            filtered_apps,
            x='Size',
            y='Rating',
            size='Installs',
            color='Translated_Category',
            title='App Size vs Average Rating (Bubble = Installs)',
            labels={'Size': 'App Size (MB)', 'Rating': 'Average Rating'},
            width=400,
            height=300
        )
        
        # Highlight 'game' category in pink
        for i, trace in enumerate(fig.data):
            if trace.name.lower() == 'game':
                fig.data[i].marker.color = 'pink'
        
        # Style settings
        fig.update_layout(
            plot_bgcolor='black',
            paper_bgcolor='black',
            font_color='white',
            title_font_size=20,
            xaxis_title_font_size=16,
            yaxis_title_font_size=16,
            margin=dict(l=40, r=40, t=40, b=40)
        )
        
        # Save chart to HTML
        fig.write_html("bubble_chart_size_rating_installs.html")
        print("✅ Chart created successfully and saved as 'bubble_chart_size_rating_installs.html'")
    else:
        print("⚠️ No data available after applying filters. Try adjusting conditions.")
else:
    print("Outside 5 PM - 7 PM IST window. Chart not displayed.")


Outside 5 PM - 7 PM IST window. Chart not displayed.


In [49]:
#You are required to create a stacked area chart to visualize the cumulative number of installs over time for each 
# app category, with each category represented as a separate color band in the chart. Apply the following filters 
# before plotting: include only apps with an average rating of at least 4.2, app names that do not contain any 
# numbers, app categories that start with the letter “T” or “P,” reviews greater than 1,000, and app sizes between 
# 20 MB and 80 MB. In the chart legend, translate “Travel & Local” into French, “Productivity” into Spanish, and 
# “Photography” into Japanese. Highlight by increasing the color intensity for any month where total installs 
# increased by more than 25% month-over-month for any category. This visualization must only be displayed between
#  4 PM IST and 6 PM IST, and it should not appear on the dashboard outside this time window.
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)
if current_time.hour >= 16 and current_time.hour < 18:
    filtered_apps = apps_df[
        (apps_df['Rating'] >= 4.2) &
        (~apps_df['App'].str.contains(r'\d', regex=True)) &
        (apps_df['Category'].str.startswith(('T', 'P'))) &
        (apps_df['Reviews'] > 1000) &
        (apps_df['Size'].between(20, 80))
    ]
    
    # Translate categories
    category_translation = {
        'Travel & Local': 'Voyage & Local',
        'Productivity': 'Productividad',
        'Photography': '写真'
    }
    
    filtered_apps['Translated_Category'] = filtered_apps['Category'].replace(category_translation)
    
    # Aggregate installs by month and category
    filtered_apps['Month'] = filtered_apps['Last Updated'].dt.to_period('M')
    monthly_installs = (
        filtered_apps
        .groupby(['Month', 'Translated_Category'])
        .agg(Total_Installs=('Installs', 'sum'))
        .reset_index()
    )
    
    monthly_installs['Month'] = monthly_installs['Month'].dt.to_timestamp()
    
    fig6 = px.area(
        monthly_installs,
        x='Month',
        y='Total_Installs',
        color='Translated_Category',
        title='Cumulative Installs Over Time by App Category',
        labels={'Total_Installs': 'Total Installs', 'Month': 'Month'},
        width=400,
        height=300
    )
    
    fig6.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font_size=20,
        xaxis_title_font_size=16,
        yaxis_title_font_size=16,
        margin=dict(l=40, r=40, t=40, b=40)
    )
    
    fig6.update_traces(marker_line_color='white', marker_line_width=1.5)
    
    save_plot_html(fig6, "stacked_area_installs_by_category_time_based.html", "cumulative installs over time by app category on play store")




In [50]:
plot_containers_split = plot_containers.split('</div>')

In [51]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [59]:
dashboard_html= """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name=viewport" content="width=device-width,initial-scale-1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify_content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container: hover .insights {{
            display: block;
        }}
        </style>
        <script>
            function openPlot(filename) {{
                window.open(filename, '_blank');
                }}
        </script>
    </head>
    <body>
        <div class= "header">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Logo_2013_Google.png/800px-Logo_2013_Google.png" alt="Google Logo">
            <h1>Google Play Store Reviews Analytics</h1>
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/78/Google_Play_Store_badge_EN.svg/1024px-Google_Play_Store_badge_EN.svg.png" alt="Google Play Store Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
    """


In [60]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [61]:
dashboard_path = os.path.join(html_file_path, "dashboard.html")
with open(dashboard_path, 'w', encoding='utf-8') as f:
    f.write(final_html)

In [62]:
webbrowser.open('file://' + os.path.realpath(dashboard_path))

True